# Routing a Complex Question Across Three Search Types

In this demo, a research agent answers a multi-hop question about the literature on retrieval-augmented generation. It breaks the question into sub-questions, and sends each one to the source that can answer it: SQL over arXiv metadata, semantic search over abstracts, or web search. Then we follow its decisions, hop by hop, in the trace.

In [1]:
import json
import sqlite3
import sys
import threading
from contextlib import closing
from pathlib import Path

import httpx
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv(".env")
sys.path.insert(0, str(Path("..").resolve()))
import web_server
from harness.agent import run_question
from harness.config import load_config
from harness.tools.base import ToolError
from research_tools import DB, WEB_URL, collection, schema

# Start local server
try:
    httpx.get(f"{WEB_URL}/search", params={"q": "test"}, timeout=2)
except httpx.HTTPError:
    web_server.Handler.index = web_server.load_index()
    server = web_server.ThreadingHTTPServer(
        (web_server.HOST, web_server.PORT), web_server.Handler
    )
    threading.Thread(target=server.serve_forever, daemon=True).start()

## 1. Three Sources

Each source answers a different kind of sub-question, and each one is a tool: a class with a `name` and a `description` the model reads, `Args`, a Pydantic model of the arguments the model fills in, and `call()`, which runs the search. The harness sends the JSON schema of `Args` to the model, and validates the model's arguments with it before calling `call()`.

Every tool also asks for two arguments it doesn't use: `sub_question`, the sub-question the call answers, and `why`, why this tool fits it. They make each routing decision visible.

Same tools are available in [research_tools.py](../research_tools.py).

### SQL over the arXiv metadata:
for counts, filters, rankings and authors. The database holds 5,732 papers from 2020 to 2025, with their authors and categories. 

The description includes the tables, so the model knows what it can query without an additional call (e.g, `get_schema` tool)

In [2]:
class SqlQuery:
    name = "sql_query"
    description = (
        "Query the metadata of arXiv papers on retrieval-augmented generation, dense retrieval "
        "and reranking: titles, abstracts, dates, categories and authors. Use it for counts, "
        "filters, rankings and author lookups. Runs one read-only SQLite query and returns at "
        f"most 50 rows, with a notice when there are more. \nThe tables:\n{schema()}"
    )

    class Args(BaseModel):
        sub_question: str = Field(description="The sub-question this call answers.")
        why: str = Field(description="Why this tool fits that sub-question.")
        sql: str = Field(description="One SELECT statement.")

    def call(self, args: Args) -> str:
        try:
            with closing(sqlite3.connect(f"file:{DB}?mode=ro", uri=True)) as db:
                cursor = db.execute(args.sql)
                columns = [c[0] for c in cursor.description]
                rows = [list(row) for row in cursor.fetchmany(51)]
        except sqlite3.Error as error:
            raise ToolError(
                f"SQL error: {error}",
                hint="fix the query using the tables in the description",
            )
        result = {"columns": columns, "rows": rows[:50]}
        if len(rows) > 50:
            result["notice"] = (
                "Only the first 50 rows are shown. Aggregate in SQL, or add a filter."
            )
        return json.dumps(result)

In [3]:
args = SqlQuery.Args(
    sub_question="How many papers are there each year?",
    why="Counting needs SQL.",
    sql="SELECT year, count(*) FROM papers GROUP BY year",
)
print(SqlQuery().call(args))

{"columns": ["year", "count(*)"], "rows": [[2020, 108], [2021, 191], [2022, 278], [2023, 484], [2024, 1693], [2025, 2978]]}


### Semantic search over the abstracts:
for methods and ideas. Each paper's title and abstract are embedded in Chroma. `arxiv_ids` lets a later hop search only the papers an earlier one found.

In [ ]:
class SemanticSearch:
    name = "semantic_search"
    description = (
        "Search the titles and abstracts of arXiv papers on retrieval-augmented generation, dense "
        "retrieval and reranking by meaning, not by exact words. Use it to find papers about a "
        "method, idea or finding, e.g. 'distilling a reranker into a dense retriever'. "
        "It can't count or list every paper: use SQL for that. "
        "Returns up to 10 of the closest papers, each with its arXiv ID, title, year, "
        "the start of its abstract, and a distance: lower is closer. "
        "To search only some papers, e.g. the ones a SQL query found, pass their arXiv IDs."
    )

    class Args(BaseModel):
        sub_question: str = Field(description="The sub-question this call answers.")
        why: str = Field(description="Why this tool fits that sub-question.")
        query: str = Field(description="What to look for, in plain words.")
        k: int = Field(5, ge=1, le=10, description="How many papers to return.")
        arxiv_ids: list[str] = Field(
            [],
            description="Only search these papers, e.g. the IDs a SQL query returned.",
        )

    def call(self, args: Args) -> str:
        where = {"arxiv_id": {"$in": args.arxiv_ids}} if args.arxiv_ids else None
        result = collection().query(
            query_texts=[args.query],
            n_results=args.k,
            where=where,
            include=["metadatas", "documents", "distances"],
        )
        papers = [
            {
                "arxiv_id": arxiv_id,
                "title": metadata["title"],
                "year": metadata["year"],
                "distance": round(distance, 3),
                "abstract": document.split("\n", 1)[1][:500],
            }
            for arxiv_id, metadata, document, distance in zip(
                result["ids"][0],
                result["metadatas"][0],
                result["documents"][0],
                result["distances"][0],
            )
        ]
        return json.dumps(papers)

In [5]:
args = SemanticSearch.Args(
    sub_question="Which papers use late interaction?",
    why="This is about a method, so semantic search fits.",
    query="late interaction retrieval with multiple vectors per document",
    k=3,
)
for paper in json.loads(SemanticSearch().call(args)):
    print(paper["distance"], paper["arxiv_id"], paper["year"], paper["title"])

0.35 2510.22215 2025 Hybrid-Vector Retrieval for Visually Rich Documents: Combining Single-Vector Efficiency and Multi-Vector Accuracy
0.352 2404.02805 2024 Efficient Multi-Vector Dense Retrieval Using Bit Vectors
0.363 2302.06589 2023 Improving Out-of-Distribution Generalization of Neural Rerankers with Contextualized Late Interaction


### Web search:
for how methods and tools are used in practice. It searches a local server with a fixed set of pages, so the results are the same on every run.

In [6]:
class WebSearch:
    name = "web_search"
    description = (
        "Search the web for how methods and tools are used in practice: libraries, frameworks, "
        "documentation and announcements. Returns pages with their URL, title and text."
    )

    class Args(BaseModel):
        sub_question: str = Field(description="The sub-question this call answers.")
        why: str = Field(description="Why this tool fits that sub-question.")
        query: str = Field(description="A web search query.")

    def call(self, args: Args) -> str:
        try:
            response = httpx.get(
                f"{WEB_URL}/search", params={"q": args.query, "k": 5}, timeout=10
            )
            response.raise_for_status()
        except httpx.HTTPError as error:
            raise ToolError(
                f"web search is unavailable: {error}",
                hint="the web server may not be running",
            )
        return response.text

In [7]:
args = WebSearch.Args(
    sub_question="Which library brings ColBERT into RAG pipelines?",
    why="Libraries and their use are on the web.",
    query="ColBERT library for RAG pipelines",
)
for page in json.loads(WebSearch().call(args))["results"][:3]:
    print(page["url"], "|", page["title"])

web_server: /search?q=ColBERT+library+for+RAG+pipelines&k=5
https://github.com/AnswerDotAI/RAGatouille | RAGatouille: easy ColBERT late-interaction retrieval for RAG pipelines
https://github.com/stanford-futuredata/ColBERT | ColBERT: reference implementation of late-interaction retrieval (ColBERTv2, PLAID)
https://github.com/AnswerDotAI/rerankers | rerankers: one Python API for cross-encoders, ColBERT, RankGPT and API rerankers


## 2. The Agent

The harness reads its config from `agent.yaml`: the model, the limits, the three tools, and the system prompt.

In [8]:
config = load_config("agent.yaml")
print("Tools:", config.tools.allow)
print(config.system_prompt.render())

Tools: ['sql_query', 'semantic_search', 'web_search']
- You are a research assistant for a literature review on retrieval-augmented generation, dense retrieval and reranking.
- First break the question into ordered sub-questions, then answer them one at a time.
- Send each sub-question to the source that can answer it. sql_query for counts, filters, rankings and authors. semantic_search for the methods and ideas in the papers. web_search for how methods and tools are used in practice.
- Use earlier results in later hops. For example, pass the arXiv IDs a SQL query found to semantic_search.
- If a result is weak or off-topic, refine the query and search again.
- In your answer, cite every claim with its source - an arXiv ID, the SQL query, or a URL.


## 3. A Multi-Hop Question



In [9]:
QUESTION = (
    "Who published the most papers in 2024? "
    "Which system or toolkit did they propose that year, and is it available as open-source software?"
)
result = run_question(
    question=QUESTION,
    config=config,
    tools=[SqlQuery(), SemanticSearch(), WebSearch()],
)
print(result.final_text)
print(
    f"\n{result.turns} turns, {result.input_tokens} input and {result.output_tokens} output tokens"
)

web_server: /search?q=FlashRAG+modular+toolkit+GitHub+repository+FlashRAG+arXiv+2405.13576&k=5
### 1) Who published the most papers in 2024 (in this RAG/dense retrieval/reranking arXiv set)?
**Zhicheng Dou** published the most, with **25** distinct 2024 papers.  
Source: SQL count/ranking over 2024 papers by author (`SELECT ... GROUP BY a.name ORDER BY paper_count DESC`)【sql_query: Identify the author(s) with the highest number of arXiv papers in the dataset in year 2024.】

---

### 2) Which system/toolkit did they propose that year?
Among Zhicheng Dou’s 2024 papers, one explicitly proposes a **toolkit**:

- **FlashRAG: A Modular Toolkit for Efficient Retrieval-Augmented Generation Research**【2405.13576】  
  Source (paper identification from Dou’s 2024 list)【sql_query: List the 2024 papers authored by the top author (Zhicheng Dou) to see which systems/toolkits they proposed that year.】 and semantic confirmation that it is a toolkit paper【2405.13576; semantic_search: Among Zhicheng Dou'

## 4. The Hops

`run_question` returns the whole conversation: each tool call the model made, and the result it got back. Here they are in order: the sub-question, why the agent chose that tool, what it sent, and the start of what came back.

In [10]:
hop = 0
for message in result.messages:
    if message.role == "assistant":
        for call in message.tool_calls:
            hop += 1
            args = dict(call.arguments)
            print(f"Hop {hop}: {call.name}")
            print(f"  sub-question: {args.pop('sub_question', '')}")
            print(f"  why:          {args.pop('why', '')}")
            print(f"  sent:         {json.dumps(args)[:300]}")
    elif message.role == "tool":
        print(f"  result:       {message.content[:300]}\n")

Hop 1: sql_query
  sub-question: Identify the author(s) with the highest number of arXiv papers in the dataset in year 2024 (retrieval-augmented generation, dense retrieval, reranking).
  why:          We need counts by author and year; SQL is best for aggregations and rankings over the arXiv metadata tables.
  sent:         {"sql": "SELECT a.name, COUNT(DISTINCT p.arxiv_id) AS paper_count\nFROM papers p\nJOIN authors a ON a.arxiv_id = p.arxiv_id\nWHERE p.year = 2024\nGROUP BY a.name\nORDER BY paper_count DESC, a.name ASC\nLIMIT 20;"}
  result:       {"columns": ["name", "paper_count"], "rows": [["Zhicheng Dou", 25], ["Guido Zuccon", 12], ["Xueqi Cheng", 12], ["Zheng Liu", 12], ["Ji-Rong Wen", 11], ["Jimmy Lin", 11], ["Shengyao Zhuang", 11], ["Fei Huang", 10], ["Kelong Mao", 10], ["Bevan Koopman", 9], ["Hamed Zamani", 9], ["Min Zhang", 9], ["Yut

Hop 2: sql_query
  sub-question: List the 2024 papers authored by the top author (Zhicheng Dou) to see which systems/toolkits they proposed t

## 5. The Trace

The same run is in Langfuse as one trace.

In [11]:
from langfuse import get_client

print(get_client().get_trace_url(trace_id=result.trace_id))

http://localhost:3000/project/cmwtj8h1o0009ry3t8e0ts4ej/traces/cd59e4130cc41b437a1ead9e7737f5c8
